In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt

# 0.2 - Process Radon

This notebooks' objective is to process the radon database : `01_OFSP_radon_bdd`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

In [ ]:
df = pd.read_excel(f'{ROOT}data/raw/01_OFSP_radon_bdd/RDN-BRIEF-1-ADMIN-CACHED-FR.2025-03-05.xlsx')

In [ ]:
list(df.columns)

In [ ]:
cols = [
    "Canton",
    "ID maison",
    "EGID",
    "Commune",
    "Catégorie de bâtiment", # Free text, ne correspond à priori pas avec une catégoristion telle que celle du RegBL
    "Année de construction", # Numérique, Variables incohérentes (1500, post-2025, 2100, ...)
    "Nombre étages",         
    "Fondation", # Attention
    "Structure de fondation", # Attention (18%)
    "Nombre étage sous-terrain", # Attention (peu d'info)
    "Terrain en pente", # Attention (peu d'info)
    "Ventilation contrôlée", # Attention (peu d'info)

    # Coordonnées 
    "LV95_ORDINATE", 
    "LV95_ABSZISSE",
    "LV03_ORDINATE",
    "LV03_ABSZISSE",
    
    "Protocole de mesure", # Pas prioritaire
    "Type de mesure", # Pas prioritaire
    "Type instrument de mesure", # Pas prioritaire

    # Date de mesure
    # Possiblement, filtrer les données sera nécessaire (>2000 only)
    # Feature eng. : fin - début = durée de la mesure (1 année complète ou 3 mois pendant la période de chauffe)
    "Début",
    "Fin",
    
    "ID local", # Pas prioritaire
    "Type de local", # Bon remplissage 
    "Séjour de personnes", # Catégorique NO = entre 0-15, YES_SHORT = 15-30, YES_LONG = >30
    "Heures de séjour par an", 
    "Etage", # Catégorisation: <-2, -2, -1, 0, 1, 2, >2


    # Valeurs cibles
    "Exposition de radon (kBqh/m3)", # (sans prise en compte du temps)
    "Incertitude (kBqh/m3)", # (sans prise en compte du temps)
    "Concentration de radon (Bq/m3)_BQ_M3", # Unique valeur cible (valeur rapportée au temps)
]

## Select only the needed columns

In [ ]:
df = df[cols]

## Add "Rdn -" to column names for nomenclature

In [ ]:
# Add 'Rdn - ' to the beginning of each column name except for 'EGID'
df.rename(columns=lambda x: 'Rdn - ' + x if x != 'EGID' else x, inplace=True)

## Compute duration column

In [ ]:
date_features = ['Rdn - Début','Rdn - Fin']

In [ ]:
for col in date_features:
    df[col] = pd.to_datetime(df[col], format="%d.%m.%Y", errors='coerce')

In [ ]:
df["Rdn - duration_days"] = (df["Rdn - Fin"] - df["Rdn - Début"]).dt.days

In [ ]:
df["Rdn - duration_days"].head()

### Transform dates in multiple columns

In [ ]:
def transform_date(df, col):
    df[col] = pd.to_datetime(df[col], errors='coerce')
    # Extract numerical features
    df[f"{col}_year"] = df[col].dt.year
    df[f"{col}_month"] = df[col].dt.month
    df[f"{col}_day"] = df[col].dt.day
    df = df.drop(columns=col)
    return df

In [ ]:
for col in date_features:
    df = transform_date(df, col)

## Save the data

In [ ]:
df.to_csv(f"{ROOT}data/processed/01_OFSP_radon_bdd.csv")